# SCIT vs AlignCoder Baseline EM & ES Evaluation Benchmark
**Target Model:** `deepseek-ai/deepseek-coder-1.3b-base` (Frozen Generator)
**Benchmark:** CrossCodeEval (100 Python + 100 Java tasks)
**Methods compared:**
1. `no_retrieval`: Closed-book in-file baseline
2. `aligncoder_baseline`: AlignCoder official retrieved prompt
3. `scit_proposed`: Submodular-Supermodular Conditional Information Transport prompt (with companion rescue and duplicate lines removed under matched budget)


In [ ]:
import os, sys, json, gzip, base64, time
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import pandas as pd

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device name:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB')


In [ ]:
EMBEDDED_B64_GZ = "..."

gz_bytes = base64.b64decode(EMBEDDED_B64_GZ)
raw_json = gzip.decompress(gz_bytes).decode('utf-8')
benchmark_tasks = json.loads(raw_json)
print(f'Successfully loaded {len(benchmark_tasks)} benchmark tasks:')
py_tasks = [x for x in benchmark_tasks if x['language'] == 'python']
ja_tasks = [x for x in benchmark_tasks if x['language'] == 'java']
print(f' - Python tasks: {len(py_tasks)}')
print(f' - Java tasks:   {len(ja_tasks)}')


In [ ]:
MODEL_NAME = 'deepseek-ai/deepseek-coder-1.3b-base'
print(f'Loading model {MODEL_NAME} in float16...')
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='cuda:0' if torch.cuda.is_available() else 'cpu',
    trust_remote_code=True
)
model.eval()
print(f'Model loaded successfully in {time.time() - t0:.1f}s!')


In [ ]:
def levenshtein(s1: str, s2: str) -> int:
    if s1 == s2:
        return 0
    if len(s1) == 0:
        return len(s2)
    if len(s2) == 0:
        return len(s1)
    v0 = list(range(len(s2) + 1))
    v1 = [0] * (len(s2) + 1)
    for i in range(len(s1)):
        v1[0] = i + 1
        for j in range(len(s2)):
            cost = 0 if s1[i] == s2[j] else 1
            v1[j + 1] = min(v1[j] + 1, v0[j + 1] + 1, v0[j] + cost)
        v0, v1 = v1, [0] * (len(s2) + 1)
    return v0[len(s2)]

def compute_metrics(prediction: str, target: str):
    gt_lines = [l.strip() for l in target.splitlines() if l.strip()]
    pred_lines = [l.strip() for l in prediction.splitlines() if l.strip()][:len(gt_lines)]
    em = 1.0 if (len(gt_lines) == len(pred_lines) and gt_lines == pred_lines) else 0.0
    gt_str = '\n'.join(gt_lines)
    pred_str = '\n'.join(pred_lines)
    max_len = max(len(gt_str), len(pred_str))
    if max_len == 0:
        es = 100.0
    else:
        dist = levenshtein(gt_str, pred_str)
        es = max(0.0, (1.0 - dist / max_len)) * 100.0
    return em, es, pred_str, gt_str

def generate_batch(prompts, max_new_tokens=64, batch_size=8):
    outputs = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i+batch_size]
        enc = tokenizer(batch, return_tensors='pt', padding=True, truncation=True, max_length=2048)
        input_ids = enc['input_ids'].to(model.device)
        attention_mask = enc['attention_mask'].to(model.device)
        prompt_len = input_ids.shape[1]
        
        with torch.no_grad():
            gen_out = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
        
        for j in range(len(batch)):
            gen_tokens = gen_out[j][prompt_len:]
            text = tokenizer.decode(gen_tokens, skip_special_tokens=True)
            outputs.append(text)
    return outputs


In [ ]:
methods = ['no_retrieval', 'aligncoder_baseline', 'scit_proposed']
prompt_map = {
    'no_retrieval': 'prompt_no_retrieval',
    'aligncoder_baseline': 'prompt_baseline',
    'scit_proposed': 'prompt_scit'
}
all_records = []

print('Starting generation and evaluation across all methods...')
t_start = time.time()

for method in methods:
    pkey = prompt_map[method]
    prompts = [task[pkey] for task in benchmark_tasks]
    print(f'Generating completions for {method} ({len(prompts)} prompts)...')
    t0 = time.time()
    preds = generate_batch(prompts, max_new_tokens=64, batch_size=8)
    print(f'Completed {method} in {time.time() - t0:.1f}s!')
    
    for task, pred in zip(benchmark_tasks, preds):
        em, es, c_pred, c_targ = compute_metrics(pred, task['target'])
        all_records.append({
            'task_id': task['task_id'],
            'dataset': task['dataset'],
            'language': task['language'],
            'method': method,
            'target': task['target'],
            'prediction': pred,
            'clean_prediction': c_pred,
            'clean_target': c_targ,
            'exact_match': em,
            'edit_similarity': es,
            'has_outside_rescue': task['has_outside_rescue'],
            'draft_had_em1': task['draft_had_em1'],
        })

print(f'All evaluations completed in {time.time() - t_start:.1f}s!')


In [ ]:
df = pd.DataFrame(all_records)

# Save full predictions and CSV summary
out_dir = Path('/kaggle/working')
df.to_json(out_dir / 'scit_eval_predictions.jsonl', orient='records', lines=True)

summary = df.groupby(['language', 'method']).agg(
    samples=('task_id', 'count'),
    exact_match=('exact_match', lambda x: f'{x.mean()*100:.2f}%'),
    edit_similarity=('edit_similarity', lambda x: f'{x.mean():.2f}'),
    rescued_exact_match=('exact_match', lambda x: f'{x[df.loc[x.index, "has_outside_rescue"]].mean()*100:.2f}%' if df.loc[x.index, "has_outside_rescue"].sum() > 0 else 'N/A')
).reset_index()

summary.to_csv(out_dir / 'scit_benchmark_summary.csv', index=False)

print('\n' + '='*85)
print('FINAL CCEVAL BENCHMARK EVALUATION RESULTS (deepseek-coder-1.3b-base)')
print('='*85)
print(summary.to_string(index=False))
print('='*85)

# Compute deltas for Python and Java
for lang in ['python', 'java']:
    sub = df[df['language'] == lang]
    b_em = sub[sub['method'] == 'aligncoder_baseline']['exact_match'].mean() * 100
    b_es = sub[sub['method'] == 'aligncoder_baseline']['edit_similarity'].mean()
    s_em = sub[sub['method'] == 'scit_proposed']['exact_match'].mean() * 100
    s_es = sub[sub['method'] == 'scit_proposed']['edit_similarity'].mean()
    print(f'\n>>> [{lang.upper()}] SCIT vs AlignCoder Baseline:')
    print(f'    - Exact Match (EM):     {s_em:.2f}% vs {b_em:.2f}%  ({s_em - b_em:+.2f}%)')
    print(f'    - Edit Similarity (ES): {s_es:.2f} vs {b_es:.2f}  ({s_es - b_es:+.2f})')
    
    # Check specifically on tasks where outside_top10 companions were rescued
    res_sub = sub[sub['has_outside_rescue']]
    res_b_em = res_sub[res_sub['method'] == 'aligncoder_baseline']['exact_match'].mean() * 100
    res_s_em = res_sub[res_sub['method'] == 'scit_proposed']['exact_match'].mean() * 100
    print(f'    - Rescue Subset EM:     {res_s_em:.2f}% vs {res_b_em:.2f}%  ({res_s_em - res_b_em:+.2f}%)')
